In [7]:
# Cell 1: Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [8]:
# Cell A2b: Download S01 and S02 into data/raw (alternative to manual upload)
import requests

SRC_URLS = {
    "mkn_directive_20_1997.pdf": "https://aipalync.org/document/download/292",
    "nadma_garis_panduan_pps_my.pdf": "https://www.nadma.gov.my/images/nadma/documents/PPSCOVID19/Garis_Panduan_PPS.pdf",
}
HEADERS = {"User-Agent": "Mozilla/5.0"}

for name, url in SRC_URLS.items():
    target = RAW_DIR / name
    if target.is_file() and target.stat().st_size > 0:
        print("Already present:", name)
        continue
    r = requests.get(url, headers=HEADERS, timeout=60)
    ok = r.status_code == 200 and r.content[:4] == b"%PDF"
    print(name, "| status:", r.status_code, "| is PDF:", ok, "| bytes:", len(r.content))
    if ok:
        target.write_bytes(r.content)
    else:
        print("  -> Download it in your browser and upload via Files > week3_project > data > raw.")

Already present: mkn_directive_20_1997.pdf
Already present: nadma_garis_panduan_pps_my.pdf


In [9]:
# Cell 3: Verify uploaded inputs (self-contained)
from pathlib import Path

RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 2
mkn_directive_20_1997.pdf bytes: 192788
nadma_garis_panduan_pps_my.pdf bytes: 230221
Upload verified. Select your relevant input(s) for Step A3.


In [21]:
# Cell A3b (optional): print context for key terms
def show_context(sid, pattern, width=140, limit=6):
    reader = PdfReader(str(select_source(relative_path=SOURCES[sid])))
    shown = 0
    for i, p in enumerate(reader.pages, 1):
        t = p.extract_text() or ""
        for m in re.finditer(pattern, t, flags=re.I):
            print(f"[{sid} p.{i}]", repr(t[max(0, m.start() - width):m.end() + width]))
            shown += 1
            if shown >= limit:
                return

show_context("S02", r"daftar|pendaftaran")
print()
show_context("S02", r"\bkit\b|agihan")
print()
show_context("S01", r"registration")

[S02 p.1] 'rga dalam kubikel yang sama.  \n \n3. Membuat penandaan jarak (sekurang -kurangnya 1 meter) yang bersesuaian di \nkawasan berikut: \ni. Kaunter pendaftaran \nii. Kaunter pengambilan makanan \niii. Ruangan menunggu di bilik air  \niv. Ruang awam (common area) \n \n4. Membuat penjarakan bersesuaian an'
[S02 p.2] 'a 1 meter dalam kalangan mangsa, kecuali \ndalam kubikel;  \nv. Marking penjarakan 1  meter dapat dilihat dan dipatuhi sewaktu beratur untuk \npendaftaran dan pengambilan makanan; \nvi. Hand sanitizer diletakkan di tempat tumpuan; \nvii. Mangsa di PPS dikehendaki memakai topeng muka setiap masa,'
[S02 p.2] 'ijalankan sebanyak 3 kali sehari, atau berdasarkan keperluan \ndan nasihat Pegawai Kesihatan KKM; dan \nxi. Mangsa di PPS digalakkan untuk mendaftar dengan aplikasi MySejahtera.  \n \n9. Sekretariat / Pengurus / Penyelia / AJK PPS perlu memastikan PPS sentiasa berada \ndalam tahap kebersiha'
[S02 p.3] 'ptional: disposable plastic apron, eye \nprotection dan boots) 

In [10]:
# Cell 4: REQUIRED Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected

    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [11]:
# CELL 4A: COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()

    # Validate source
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")

    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")

    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")

    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())

    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


In [ ]:
# Cell A3: Inspect two source PDFs (S01, S02)
!pip -q install pypdf
import re
import json
from pypdf import PdfReader

SOURCES = {
    "S01": "mkn_directive_20_1997.pdf",
    "S02": "nadma_garis_panduan_pps_my.pdf",
}
TERMS = {
    "evacuation centre(s)": r"evacuation cent(?:re|er)s?",
    "Pusat Pemindahan Sementara": r"pusat pemindahan sementara",
    "PPS": r"\bpps\b",
    "registration": r"registration",
    "pendaftaran": r"pendaftaran",
    "daftar": r"\bdaftar\b",
}
EVIDENCE_TERM = {"S01": r"registration", "S02": r"daftar"}
STRAY_E = r"(?<=[a-z])E(?=\s|[a-z])"

inspection = {}
for sid, name in SOURCES.items():
    f = select_source(relative_path=name)
    reader = PdfReader(str(f))
    pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]
    text = "\n".join(t for _, t in pages)
    empty_pages = [n for n, t in pages if not t.strip()]
    counts = {k: len(re.findall(v, text, flags=re.I)) for k, v in TERMS.items()}
    first_page, first_text = next((n, t) for n, t in pages if t.strip())
    stray = [text[max(0, m.start() - 12):m.end() + 8] for m in re.finditer(STRAY_E, text)]

    evidence_page, evidence = None, None
    for n, t in pages:
        m = re.search(EVIDENCE_TERM[sid], t, flags=re.I)
        if m:
            evidence_page = n
            evidence = t[max(0, m.start() - 150):m.end() + 250]
            break

    inspection[sid] = {
        "file": str(f.relative_to(RAW_DIR)),
        "sha256": source_hash(f),
        "pages": len(pages),
        "characters": len(text),
        "pages_without_text": empty_pages,
        "term_counts": counts,
        "possible_stray_E_artefacts": len(stray),
    }

    print(f"--- {sid}: {f.name} ---")
    print("SHA-256:", inspection[sid]["sha256"])
    print("Pages:", len(pages), "| Characters:", len(text))
    print("Pages without extractable text:", empty_pages)
    print("Term counts:", counts)
    print("Possible stray 'E' artefacts:", len(stray), stray[:3])
    print(f"Sample 1 (page {first_page}):", repr(first_text[:300]))
    print(f"Sample 2, registration evidence (page {evidence_page}):", repr(evidence))
    print()
    assert text.strip(), f"No extractable text in {name}"

print(json.dumps(inspection, indent=2, ensure_ascii=False))

In [14]:
# Cell B0: Part B helpers for Pathway A (two PDF sources, page-aware)
# Run after Cell 4 and Cell 4A.
!pip -q install pypdf
import re
import json
import unicodedata
from pypdf import PdfReader

SOURCES = {
    "S01": "mkn_directive_20_1997.pdf",
    "S02": "nadma_garis_panduan_pps_my.pdf",
}
CHUNK_SIZE = 500
# S02 page 4 is a flowchart whose reading order is unreliable: keep it as one chunk
WHOLE_PAGES = {"S02": [4]}
# Pages excluded as table of contents. Page 1 is detected automatically; page 2 is not,
# because pypdf returns it as one long line, so it is listed explicitly.
TOC_PAGES = {"S01": [1, 2]}
COVID_TERMS = r"covid|saringan kesihatan|topeng muka|mysejahtera|thermometer|disinfeksi|hand sanitizer"

# Cleaning rules, each justified by Part A inspection evidence
QUOTE_FIX = re.compile(r"(‘[^‘’\n]{1,40}?[a-z])E(?=[a-z])")   # 'post-mortemEis' -> 'post-mortem’ is'
STRAY_E = r"(?<=[a-z])E(?=\s|[a-z])"                           # used only to count remaining artefacts
LIST_START = re.compile(
    r"^(?:\(?\d{1,2}[.)]\s|\(?[a-z][.)]\s|[ivx]{1,4}[.)]\s|[A-Z]\s?:\s|[•\-]\s)"
)

def extract_pages(sid):
    f = select_source(relative_path=SOURCES[sid])
    reader = PdfReader(str(f))
    return f, [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]

def normalise(text):
    text = unicodedata.normalize("NFKC", text)                  # fixes ligatures such as the fi in "Definition"
    text = QUOTE_FIX.sub(r"\1’ ", text)                         # repairs stray E for closing quote
    text = re.sub(r"(?<=[A-Za-z]) -(?=[A-Za-z0-9])", "-", text) # "COVID -19" -> "COVID-19"
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def reflow(text):
    """Join wrapped lines into paragraphs; keep list items and headings separate."""
    paragraphs, current = [], ""
    for raw in text.split("\n"):
        line = raw.strip()
        if not line:
            if current:
                paragraphs.append(current)
                current = ""
            continue
        if not current:
            current = line
            continue
        starts_new = (
            bool(LIST_START.match(line))
            or (current[-1] in ".:;" and line[0].isupper())
            or (line.isupper() and len(line) > 3)
            or current.isupper()
        )
        if starts_new:
            paragraphs.append(current)
            current = line
        else:
            current += " " + line
    if current:
        paragraphs.append(current)
    return "\n\n".join(paragraphs)

def is_toc(page_text):
    lines = [l.strip() for l in page_text.split("\n") if l.strip()]
    if "TABLE OF CONTENTS" in page_text.upper():
        return True
    if len(lines) < 5:
        return False
    hits = sum(1 for l in lines if len(l) < 70 and re.search(r"\s\d{1,3}$", l))
    return hits / len(lines) > 0.5

# Chunkers (fixed and paragraph-aware are the lab versions)
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

def page_aware_chunks(sid, pages, size=CHUNK_SIZE):
    """Chunk page by page so every chunk keeps its PDF page number."""
    rows, skipped = [], []
    for page_no, raw in pages:
        if page_no in TOC_PAGES.get(sid, []) or is_toc(raw):
            skipped.append(page_no)
            continue
        text = normalise(raw)
        if not text:
            continue
        if page_no in WHOLE_PAGES.get(sid, []):
            rows.append({"page": page_no, "text": re.sub(r"\s+", " ", text), "chunk_type": "flowchart_page"})
            continue
        for chunk in paragraph_chunks(reflow(text), size):
            rows.append({"page": page_no, "text": chunk, "chunk_type": "text"})
    return rows, skipped

print("Cell B0 ready: extract_pages, normalise, reflow, fixed_chunks, paragraph_chunks, page_aware_chunks")

Cell B0 ready: extract_pages, normalise, reflow, fixed_chunks, paragraph_chunks, page_aware_chunks


In [15]:
# Cell 5A1: Pathway A1 comparison for both sources
# Compares the lab's fixed and paragraph-aware chunking with a page-aware version.
def stats(chunks):
    lens = [len(c) for c in chunks]
    ends = sum(1 for c in chunks if c.rstrip()[-1:] in ".:;!?)")
    return len(chunks), round(sum(lens) / len(lens)), min(lens), max(lens), round(100 * ends / len(chunks))

def first_with(chunks, pattern):
    return next((c for c in chunks if re.search(pattern, c, re.I)), None)

EVIDENCE_TERM = {"S01": r"registration", "S02": r"daftar"}
comparison = {}

for sid in SOURCES:
    f, pages = extract_pages(sid)
    text = "\n".join(t for _, t in pages)
    clean = re.sub(r"[ \t]+", " ", text).strip()          # lab cleaning
    assert clean, "No usable text: check extraction or OCR needs."

    fixed = fixed_chunks(clean, CHUNK_SIZE)
    para = paragraph_chunks(clean, CHUNK_SIZE)
    rows, skipped = page_aware_chunks(sid, pages, CHUNK_SIZE)
    page_chunks = [r["text"] for r in rows]
    midword = sum(
        1 for i in range(CHUNK_SIZE, len(clean), CHUNK_SIZE)
        if clean[i - 1].isalnum() and clean[i].isalnum()
    )

    comparison[sid] = {
        "fixed": stats(fixed),
        "paragraph-aware": stats(para),
        "page-aware": stats(page_chunks),
        "fixed_midword_cuts": midword,
        "skipped_toc_pages": skipped,
    }

    print(f"=== {sid}: {f.name} | pages: {len(pages)} | cleaned chars: {len(clean)} ===")
    print("Config           | chunks | mean | min | max | % ending at punctuation | page recorded")
    for name, chunks, has_page in [
        ("fixed-500", fixed, "no"),
        ("paragraph-aware", para, "no"),
        ("page-aware", page_chunks, "yes"),
    ]:
        n, mean, lo, hi, pct = stats(chunks)
        print(f"{name:<16} | {n:>6} | {mean:>4} | {lo:>3} | {hi:>3} | {pct:>23} | {has_page}")
    print("Fixed chunks cut inside a word:", midword)
    print("Pages excluded as table of contents:", skipped)

    term = EVIDENCE_TERM[sid]
    print(f"\nChunk containing '{term}' under each configuration:")
    for name, chunks in [("fixed", fixed), ("paragraph-aware", para), ("page-aware", page_chunks)]:
        hit = first_with(chunks, term)
        print(f"[{name}]", repr(hit[:260]) if hit else None)
    print()

print("PATHWAY A1 completed successfully.")

=== S01: mkn_directive_20_1997.pdf | pages: 26 | cleaned chars: 55055 ===
Config           | chunks | mean | min | max | % ending at punctuation | page recorded
fixed-500        |    111 |  496 |  55 | 500 |                       6 | no
paragraph-aware  |    136 |  404 |   5 | 500 |                      33 | no
page-aware       |    138 |  381 |  21 | 500 |                      72 | yes
Fixed chunks cut inside a word: 66
Pages excluded as table of contents: [1, 2]

Chunk containing 'registration' under each configuration:
[fixed] '.\n3. To carry out registration on the victim for the purpose of rehabilitation.\n4. To offer guidance, advice/counselling to victims.\nC: MALAYSIAN PEOPLE VOLUNTARY ALLIANCE(RELA)\n1. To assist with the evacuation of victims.\n2. To assist with preparing /distribu'
[paragraph-aware] 'res.\n2. To make an arrangement and distributing food, clothings and other necessities.\n3. To carry out registration on the victim for the purpose of rehabilitation.\n4. To off

In [16]:
# Cell 5A2: Pathway A2 export for both sources (page-aware records)
import json

CHOSEN_METHOD = "page-aware"   # alternatives from A1: "fixed" or "paragraph-aware" (not used: no page numbers)
assert CHOSEN_METHOD == "page-aware", "This cell exports the page-aware configuration"
approach = f"page-aware-paragraph-char-{CHUNK_SIZE}"

records = []
SOURCE_FILES = {}
PROCESSING = {}

for sid in SOURCES:
    f, pages = extract_pages(sid)
    sha = source_hash(f)
    SOURCE_FILES[sid] = f
    rows, skipped = page_aware_chunks(sid, pages, CHUNK_SIZE)
    for n, row in enumerate(rows, 1):
        records.append({
            "record_id": f"{sid}-p{row['page']:02d}-{n:03d}",
            "text": row["text"],
            "source_id": sid,
            "source": str(f.relative_to(RAW_DIR)),
            "source_sha256": sha,
            "page": row["page"],
            "chunk_type": row["chunk_type"],
            "flags": ["covid_health"] if re.search(COVID_TERMS, row["text"], re.I) else [],
            "processing": approach,
        })
    PROCESSING[sid] = (
        f"{approach}; NFKC normalisation, stray-E quote repair, spaced-hyphen repair, line reflow; "
        f"table-of-contents pages excluded: {skipped}; "
        f"whole-page chunks: {WHOLE_PAGES.get(sid, [])}"
    )

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

# Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(r["text"].strip() for r in records), "One or more chunks are empty"
assert all(r["page"] is not None for r in records), "A chunk has no page number"
assert len({r["record_id"] for r in records}) == len(records), "Duplicate record_id"
assert {r["source_id"] for r in records} == set(SOURCES), "A source produced no records"

with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
assert len(exported_records) == len(records), "Exported record count does not match"

# Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
for sid in SOURCES:
    rs = [r for r in records if r["source_id"] == sid]
    pages_used = sorted({r["page"] for r in rs})
    flagged = sum(1 for r in rs if r["flags"])
    print(f"{sid}: {len(rs)} chunks | pages with chunks: {pages_used} | covid_health flagged: {flagged}")
    print("   processing:", PROCESSING[sid])
remaining = {
    sid: len(re.findall(STRAY_E, " ".join(r["text"] for r in records if r["source_id"] == sid)))
    for sid in SOURCES
}
print("Possible stray 'E' artefacts remaining after cleaning:", remaining)
print("Prepared file:", prepared_path, "| bytes:", prepared_path.stat().st_size)
print("Validation: PASS")
print("First S01 record:", next(r for r in records if r["source_id"] == "S01"))
print("First S02 record:", next(r for r in records if r["source_id"] == "S02"))

PATHWAY A2 completed successfully.
Selected approach: page-aware-paragraph-char-500
Total chunks exported: 156
S01: 138 chunks | pages with chunks: [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26] | covid_health flagged: 0
   processing: page-aware-paragraph-char-500; NFKC normalisation, stray-E quote repair, spaced-hyphen repair, line reflow; table-of-contents pages excluded: [1, 2]; whole-page chunks: []
S02: 18 chunks | pages with chunks: [1, 2, 3, 4, 5] | covid_health flagged: 13
   processing: page-aware-paragraph-char-500; NFKC normalisation, stray-E quote repair, spaced-hyphen repair, line reflow; table-of-contents pages excluded: []; whole-page chunks: [4]
Possible stray 'E' artefacts remaining after cleaning: {'S01': 0, 'S02': 0}
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl | bytes: 104317
Validation: PASS
First S01 record: {'record_id': 'S01-p03-001', 'text': 'JBPM : Malaysian Fire and Rescue Department JPBB

In [17]:
# B3 basic output contract plus content spot-check
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

with prepared_path.open("r", encoding="utf-8") as file:
    rows = [json.loads(line) for line in file if line.strip()]
for sid, term in [("S01", r"registration|register"), ("S02", r"daftar|pendaftaran")]:
    hits = [r for r in rows if r["source_id"] == sid and re.search(term, r["text"], re.I)]
    print(f"\n{sid}: {len(hits)} chunks mention {term!r}")
    for r in hits[:2]:
        print(f"  {r['record_id']} (page {r['page']}, {r['chunk_type']}):", repr(r["text"][:240]))

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 104317
Validation: file exists and is non-empty; verify content manually too.

S01: 2 chunks mention 'registration|register'
  S01-p25-133 (page 25, text): '2. To make an arrangement and distributing food, clothings and other necessities.\n\n3. To carry out registration on the victim for the purpose of rehabilitation.\n\n4. To offer guidance, advice/counselling to victims.\n\nC: MALAYSIAN PEOPLE VOLU'
  S01-p26-137 (page 26, text): '3. To provide other telecommunication services needed for the operation by the relevant agencies.\n\n4. To place enough officers and staffs to give counselling services and maintenance services on the telecommunication lines and other equipme'

S02: 8 chunks mention 'daftar|pendaftaran'
  S02-p01-003 (page 1, text): '1. Sekretariat JPBN/D perlu memastikan PPS mempunyai fasiliti asas yang berfungsi dengan baik dan kondusif bagi memastikan arahan PKP semasa dapat dipatuhi , terutamanya me

In [18]:
# B4.1: Register Pathway A (both sources map to the same prepared file)
for sid, f in SOURCE_FILES.items():
    register_lineage(
        source=f,
        output=prepared_path,
        processing=PROCESSING[sid]
    )
print("Registered entries:", len(lineage_entries))

Registered: mkn_directive_20_1997.pdf -> prepared_text.jsonl
Registered: nadma_garis_panduan_pps_my.pdf -> prepared_text.jsonl
Registered entries: 2


In [19]:
# CELL 6A: GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)

lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
    "Saved to:",
    BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 2
Saved to: /content/week3_project/data/week3_manifest.json


In [20]:
# CELL 6B: VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
    "Common lineage manifest not found."
)

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])
assert saved_records, (
    "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]

    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 2
Manifest: /content/week3_project/data/week3_manifest.json


In [22]:
# Cell 7: REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}

# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(
                f"Source folder not found: {source}"
            )
    else:
        if not source.is_file():
            raise FileNotFoundError(
                f"Source file not found: {source}"
            )
    return source

# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file()
            and p.suffix.lower() in {
                ".png", ".jpg", ".jpeg", ".webp"
            }
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError(
            "No valid source files to ingest."
        )

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(
        inputs,
        output_dir,
        configuration
    )
    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError(
            "No prepared outputs were produced."
        )
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(
                f"Prepared output missing: {output}"
            )
        if output.stat().st_size == 0:
            raise ValueError(
                f"Prepared output is empty: {output}"
            )

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [
            sha256_file(p) for p in outputs
        ],
        "output_bytes": [
            p.stat().st_size for p in outputs
        ]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [
            {
                "path": str(p),
                "sha256": sha256_file(p)
            }
            for p in inputs
        ],
        "outputs": [
            str(p) for p in outputs
        ],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": (
            "Not verified: check before sharing"
        )
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(
        json.dumps(
            record,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        f"SUCCESS: {kind} | "
        f"Inputs: {len(inputs)} | "
        f"Outputs: {len(outputs)}"
    )
    print("Manifest:", manifest_path)
    return record

# Step 5: Register and run a selected pathway
def register_and_run(
    kind,
    source,
    inspect_fn,
    prepare_fn,
    configuration
):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(
            kind,
            source,
            inspect_fn,
            prepare_fn,
            configuration
        )

    first = runner()
    PIPELINE_RUNS[kind] = {
        "first": first,
        "runner": runner
    }
    return first

print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [23]:
# Cell 8A: CONDITIONAL REQUIRED: PDF/TXT projects (two sources, page-aware)
# Requires Cell 7 (Common V1 Pipeline Utilities) and Cell B0 (Part B helpers).
!pip -q install pypdf
import re
import json
from pypdf import PdfReader

APPROACH_A = f"page-aware-paragraph-char-{CHUNK_SIZE}"

# Step 1: Configuration, identical to the Part B decisions
def make_config(sid):
    return {
        "source_id": sid,
        "method": "page-aware-paragraph",
        "chunk_size": CHUNK_SIZE,
        "toc_pages": TOC_PAGES.get(sid, []),
        "whole_pages": WHOLE_PAGES.get(sid, []),
        "cleaning": "NFKC; stray-E quote repair; spaced-hyphen repair; line reflow",
    }

def read_pages(src):
    if src.suffix.lower() != ".pdf":
        raise ValueError("Expected a PDF source")
    reader = PdfReader(str(src))
    return [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]
    pages = read_pages(src)
    text = "\n".join(t for _, t in pages)
    if not text.strip():
        raise ValueError(
            "No extractable text found. "
            "Scanned PDFs may require OCR."
        )
    return {
        "source": src.name,
        "pages": len(pages),
        "characters": len(text),
        "pages_without_text": [n for n, t in pages if not t.strip()],
        "toc_pages_excluded": cfg["toc_pages"],
        "whole_page_chunks": cfg["whole_pages"],
    }

# Step 3: Prepare and store traceable page-aware records
def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    sid = cfg["source_id"]
    pages = read_pages(src)
    rows, skipped = page_aware_chunks(sid, pages, int(cfg["chunk_size"]))
    if not rows:
        raise ValueError("No text chunks produced")

    sha = sha256_file(src)
    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for n, row in enumerate(rows, 1):
            record = {
                "record_id": f"{sid}-p{row['page']:02d}-{n:03d}",
                "text": row["text"],
                "source_id": sid,
                "source": str(src.relative_to(RAW_DIR)),
                "source_sha256": sha,
                "page": row["page"],
                "chunk_type": row["chunk_type"],
                "flags": ["covid_health"] if re.search(COVID_TERMS, row["text"], re.I) else [],
                "processing": APPROACH_A,
            }
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    mapping = [{"source": str(src), "output": str(target)}]
    method = (
        f"{APPROACH_A}; table-of-contents pages excluded: {skipped}; "
        f"whole-page chunks: {cfg['whole_pages']}"
    )
    return [target], mapping, method

# Step 4: Execute and register one run per source
SOURCE_A1 = resolve_source(SOURCES["S01"])
SOURCE_A2 = resolve_source(SOURCES["S02"])

run_A1 = register_and_run(
    "documents_s01", SOURCE_A1, inspect_documents, prepare_documents, make_config("S01")
)
run_A2 = register_and_run(
    "documents_s02", SOURCE_A2, inspect_documents, prepare_documents, make_config("S02")
)

print("Cell 8A completed successfully.")
print("Inspection S01:", run_A1["inspection"])
print("Inspection S02:", run_A2["inspection"])

SUCCESS: documents_s01 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s01/pipeline_manifest.json
SUCCESS: documents_s02 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s02/pipeline_manifest.json
Cell 8A completed successfully.
Inspection S01: {'source': 'mkn_directive_20_1997.pdf', 'pages': 26, 'characters': 55055, 'pages_without_text': [], 'toc_pages_excluded': [1, 2], 'whole_page_chunks': []}
Inspection S02: {'source': 'nadma_garis_panduan_pps_my.pdf', 'pages': 5, 'characters': 7586, 'pages_without_text': [], 'toc_pages_excluded': [], 'whole_page_chunks': [4]}


In [24]:
# Cell 8A-check: compare Part C outputs with the Part B export and inspect records
part_b = (PROCESSED_DIR / "prepared_text.jsonl").read_text(encoding="utf-8")
part_c = "".join(
    (PIPE_DIR / k / "prepared_text.jsonl").read_text(encoding="utf-8")
    for k in ("documents_s01", "documents_s02")
)
print("Part C outputs identical to Part B export:", part_b == part_c)

for k in ("documents_s01", "documents_s02"):
    path = PIPE_DIR / k / "prepared_text.jsonl"
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"\n{k}: {len(rows)} records | bytes: {path.stat().st_size}")
    print("Representative record:", rows[len(rows) // 2])

Part C outputs identical to Part B export: True

documents_s01: 138 records | bytes: 91640
Representative record: {'record_id': 'S01-p12-070', 'text': 'TRAINING AND COURSES\n\n38. Chance to attend any programme of development disaster and training that able to be applied with the disaster management and control should be given to officers for concerned whether in local or outside the country. Training and courses is necessary to upgrade the disaster management and control among the officers. BKN and related agencies are responsible in creating their own training courses of planning and performing if necessary.', 'source_id': 'S01', 'source': 'mkn_directive_20_1997.pdf', 'source_sha256': '1f9b2faf796291454d741a1b7d66abd3cebf7ed3132f2bef7ecda675ac336ee4', 'page': 12, 'chunk_type': 'text', 'flags': [], 'processing': 'page-aware-paragraph-char-500'}

documents_s02: 18 records | bytes: 12677
Representative record: {'record_id': 'S02-p02-010', 'text': '9. Sekretariat / Pengurus / Penyelia / 

In [25]:
# Cell 9: REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A-8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
    "No pipeline runs registered. "
    "Run at least one selected Cell 8 pathway first."
)

print("--- Repeatability Verification ---")

# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]

    # Execute the same pipeline again
    second = entry["runner"]()

    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], (
        f"Source fingerprint changed for {kind}"
    )

    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], (
        f"Prepared output changed for {kind}"
    )

    # Step 5: Verify source-to-output mapping
    assert (
        first["source_output_mapping"]
        == second["source_output_mapping"]
    ), (
        f"Source-to-output mapping changed for {kind}"
    )

    print(
        f"PASS: {kind} | source fingerprints, "
        "prepared outputs and mappings match"
    )

# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents_s01 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s01/pipeline_manifest.json
PASS: documents_s01 | source fingerprints, prepared outputs and mappings match
SUCCESS: documents_s02 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_s02/pipeline_manifest.json
PASS: documents_s02 | source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
